In [1]:
import sympy as sp
import sys
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir
from affine_body_core import compute_J_vec

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")

In [2]:
kappa  = Eigen.Scalar("kappa")

# Basis vectors
Vi_bar = Eigen.Vector("Vi_bar",3)
Ni_bar = Eigen.Vector("Ni_bar",3)
Vj_bar = Eigen.Vector("Vj_bar",3)
Nj_bar = Eigen.Vector("Nj_bar",3)

# Affine Body DOF vectors
qi = Eigen.Vector("qi",12)
qj = Eigen.Vector("qj",12)



$$
E:=\frac{1}{2} K \sin^2(\theta- \tilde \theta) \\
  =\frac{1}{2}K\left(\sin \theta \cos \tilde\theta - \cos \theta \sin \tilde \theta \right)^2
$$

$$
\mathbf{F}_{01} = \begin{bmatrix}
\mathbf{bar{\mathbf{V}}}_i \\ \mathbf{N}_i \\
\mathbf{bar{\mathbf{V}}}_j \\ \mathbf{N}_j \\
\end{bmatrix}_{12 \times 1}
$$

Frame Affine Body Mapping:

$$
\mathbf{J}_{01} = \begin{bmatrix}
\mathbf{J}(\bar{\mathbf{V}}_i) & 0 \\
\mathbf{J}(\bar{\mathbf{N}}_i) & 0 \\
0 & \mathbf{J}(\bar{\mathbf{V}}_j) \\
0 & \mathbf{J}(\bar{\mathbf{N}}_j) \\
\end{bmatrix}_{12 \times 24}
$$

$$
\mathbf{F}_{01} = \mathbf{J}_{01} \cdot
\begin{bmatrix}
\mathbf{q}_i \\
\mathbf{q}_j \\
\end{bmatrix}
$$

In [3]:
# Mapping
J01 = sp.Matrix.zeros(12,24)
J01[0:3, 0:12] = compute_J_vec(Vi_bar)
J01[3:6, 0:12] = compute_J_vec(Ni_bar)
J01[6:9, 12:24] = compute_J_vec(Vj_bar)
J01[9:12, 12:24] = compute_J_vec(Nj_bar)

content = ""

# from ABD q to F
F0_q = J01 @ sp.Matrix.vstack(qi, qj)
Cl_F01 = Gen.Closure(Vi_bar, Ni_bar, qi, # Affine Body i
                     Vj_bar, Nj_bar, qj) # Affine Body j

# from F Gradient to ABD Gradient
G12 = Eigen.Vector("G12",12)
J01T_G01 = J01.T @ G12
Cl_G01 = Gen.Closure(G12, 
                     Vi_bar, Ni_bar, # Affine Body i
                     Vj_bar, Nj_bar) # Affine Body j
# from F Hessian to ABD Hessian
H12x12 = Eigen.Matrix("H12x12",12,12)
J01T_H01_J01 = J01.T @ H12x12 @ J01
Cl_H01 = Gen.Closure(H12x12, 
                     Vi_bar, Ni_bar, # Affine Body i
                     Vj_bar, Nj_bar) # Affine Body j

content += f""" 
// Mapping between ABD qi qj to F01

{Cl_F01("F01_q", F0_q)}
{Cl_G01("J01T_G01", J01T_G01)}
{Cl_H01("J01T_H01_J01", J01T_H01_J01)}  

"""     

F01_q = Eigen.Vector("F01_q", 12)
Vi = F01_q[0:3,0]
Ni = F01_q[3:6,0]
Vj = F01_q[6:9,0]
Nj = F01_q[9:12,0]

cos_theta = (Vi.dot(Vj) + Ni.dot(Nj)) / 2
sin_theta = (Vj.dot(Ni) - Nj.dot(Vi)) / 2

theta_tilde = Eigen.Scalar("theta_tilde")
cos_tilde = cos(theta_tilde[0, 0])
sin_tilde = sin(theta_tilde[0, 0])
# Angle
currAngle = atan2(sin_theta, cos_theta)
CL_Angle = Gen.Closure(F01_q)
content += f""" //Revolute Joint Angle

{CL_Angle("currAngle", currAngle)}
// -------------------------------------------------------------------------
"""

E = 0.5 * kappa * (sin_theta * cos_tilde - cos_theta * sin_tilde)**2
Cl_E01 = Gen.Closure(kappa, F01_q, theta_tilde)

dE01dF01 = VecDiff(E, F01_q)
ddE01ddF01 = VecDiff(dE01dF01, F01_q)

content += f""" // Drving Revolute Joint Energy and Derivatives(Gradient & Hessian)

{Cl_E01("E", E)}
{Cl_E01("dEdF01", dE01dF01)}
{Cl_E01("ddEddF01", ddE01ddF01)}

// ---------
"""

In [4]:
f = open(backend_source_dir('cuda') / 'affine_body/constitutions/sym/affine_body_driving_revolute_joint.inl', 'w')
f.write(content)
f.close()